# Каталог всех задач re-rl

Этот ноутбук — живой справочник по **всем типам задач** библиотеки. Он ничего не хардкодит:
категории и примеры строятся автоматически из реестра `registry` и генераторов
`ALL_TASK_GENERATORS`, поэтому при добавлении новых задач ноутбук обновляется сам.

**Что внутри:**
1. Сводка по категориям (сколько типов в каждой области).
2. Математика — краткий пример на **каждый** тип (условие + ответ + проверка `verify`).
3. Логика и ризонинг — «spotlight» с полным пошаговым рассуждением.
4. Физика — краткий пример на каждый тип.
5. Генерация датасета по выбранной категории.
6. Как развернуть любую задачу целиком.

Группировка по категориям выполняется по модулю класса
(`re_rl.tasks.<домен>.<категория>....`).

## Настройка и вспомогательные функции

In [ ]:
import warnings
from collections import defaultdict

warnings.filterwarnings("ignore")

from re_rl.tasks.generators import ALL_TASK_GENERATORS
from re_rl.tasks.physics.generators import ALL_PHYSICS_TASK_GENERATORS
from re_rl.tasks.registry import registry

# Человекочитаемые названия категорий (при отсутствии — берётся имя модуля).
PRETTY = {
    "abstract_algebra": "Абстрактная алгебра", "algebra": "Алгебра", "analysis": "Матанализ",
    "applied": "Прикладная математика", "discrete": "Дискретная математика",
    "geometry": "Геометрия", "linear_algebra": "Линейная алгебра",
    "logic": "Логика и ризонинг", "planning": "Планирование и поиск",
    "probability": "Вероятность и статистика",
    "astrophysics": "Астрофизика", "electricity": "Электричество", "fluids": "Гидродинамика",
    "magnetism": "Магнетизм", "measurements": "Измерения и погрешности",
    "mechanics": "Механика", "networks": "Масштабируемые сети", "nuclear": "Ядерная физика",
    "oscillations": "Колебания", "quantum": "Квантовая физика",
    "relativity": "Теория относительности", "thermodynamics": "Термодинамика",
    "waves": "Волны и оптика", "lean_proof_task": "Формальная математика (Lean 4)",
}
DOMAIN_LABEL = {"math": "МАТЕМАТИКА", "physics": "ФИЗИКА", "formal": "ФОРМАЛЬНАЯ МАТЕМАТИКА"}


def build_groups():
    """{(домен, категория): [имена задач]} — по модулю зарегистрированного класса."""
    groups = defaultdict(list)
    for name in ALL_TASK_GENERATORS:
        cls = registry.get(name)
        parts = (cls.__module__ if cls else "").split(".")
        domain, sub = (parts[2], parts[3]) if len(parts) >= 4 else ("other", "other")
        groups[(domain, sub)].append(name)
    return groups


def make_example(name, language="ru", difficulty=4, reasoning=True):
    """Единый безопасный способ сгенерировать и решить задачу любого типа."""
    gen = ALL_TASK_GENERATORS[name]
    try:
        task = gen(language=language, difficulty=difficulty, reasoning_mode=reasoning)
    except TypeError:  # часть физических генераторов не принимает reasoning_mode
        task = gen(language=language, difficulty=difficulty)
    try:
        task.get_result()  # гарантируем solve()
    except Exception:
        pass
    if getattr(task, "final_answer", None) is None:
        task.solve()
    return task


def _one_line(text, limit):
    s = " ".join(str(text).split())
    return s if len(s) <= limit else s[:limit] + "…"


def brief(name, difficulty=4):
    """Краткая карточка: условие, ответ, отметка самопроверки verify()."""
    t = make_example(name, difficulty=difficulty)
    try:
        ok = t.verify(f"<answer>{t.final_answer}</answer>") == 1.0
    except Exception:
        ok = False
    print(f"[{'✓' if ok else ' '}] {name}")
    print(f"      Q: {_one_line(t.description, 200)}")
    print(f"      A: {_one_line(t.final_answer, 90)}")


def show_domain(domain, difficulty=4):
    groups = build_groups()
    for key in sorted(groups):
        d, sub = key
        if d != domain:
            continue
        names = sorted(groups[key])
        print("=" * 72)
        print(f"{PRETTY.get(sub, sub)}  ({len(names)} типов)")
        print("=" * 72)
        for n in names:
            brief(n, difficulty=difficulty)
        print()


def show_full(name, language="ru", difficulty=5):
    """Полный вывод: условие, все шаги решения, ответ."""
    t = make_example(name, language=language, difficulty=difficulty)
    print("#" * 72)
    print(f"{name}  (сложность {difficulty})")
    print("#" * 72)
    print(t.description)
    print("-" * 30, "решение", "-" * 30)
    for s in t.solution_steps:
        print(s)
    print("ОТВЕТ:", t.final_answer)
    print()


print("Готово. Доступно типов задач:", len(ALL_TASK_GENERATORS))

## 1. Сводка по категориям

In [ ]:
groups = build_groups()
total = len(ALL_TASK_GENERATORS)
n_phys = len(ALL_PHYSICS_TASK_GENERATORS)

for domain in ("math", "formal", "physics"):
    subs = sorted(k for k in groups if k[0] == domain)
    if not subs:
        continue
    dom_total = sum(len(groups[k]) for k in subs)
    print(f"\n{DOMAIN_LABEL.get(domain, domain.upper())}  —  {dom_total} типов")
    print("-" * 56)
    for key in subs:
        print(f"  {PRETTY.get(key[1], key[1]):<34} {len(groups[key]):>3}")

print("\n" + "=" * 56)
print(f"ИТОГО: {total} типов  =  {total - n_phys} математических + {n_phys} физических")
print("=" * 56)

## 2. Математика — пример на каждый тип

Для каждого типа: условие (`Q`), финальный ответ (`A`) и отметка `[✓]`, если задача
проходит самопроверку `verify()` собственного ответа.

In [ ]:
show_domain("math")
show_domain("formal")

## 3. Логика и ризонинг — полное рассуждение

Свежая волна ризонинг-задач с пошаговым решением: прямой вывод (Datalog), унификация
термов, β-редукция λ-исчисления, резолюция, ДП на подпоследовательностях, KMP и пятнашки.

In [ ]:
for name in ["datalog_inference", "unification", "lambda_calculus", "resolution",
             "lis_dp", "kmp_matching", "sliding_puzzle"]:
    show_full(name, difficulty=5)

## 4. Физика — пример на каждый тип

In [ ]:
show_domain("physics")

## 5. Генерация датасета по категории

Любую категорию можно превратить в SFT-датасет, передав список её типов в
`DatasetGenerator`. Ниже — пример для ризонинг-задач; чтобы взять всю категорию целиком,
используйте `sorted(build_groups()[("math", "logic")])` и т.п.

In [ ]:
from re_rl.dataset_generator import DatasetGenerator

reasoning_types = ["datalog_inference", "unification", "lambda_calculus", "resolution",
                   "lis_dp", "kmp_matching", "sliding_puzzle"]

gen = DatasetGenerator()
dataset = gen.generate_sft_dataset(
    task_types=reasoning_types,
    num_samples=21,
    language="ru",
    difficulties=[3, 5, 7],
    reasoning_mode=True,
)

print(f"Сгенерировано примеров: {len(dataset)}")
print("=" * 60)
print("input:", dataset[0]["input"][:200])
print("-" * 60)
print("output:\n", dataset[0]["output"][:500])

## 6. Развернуть любую задачу целиком

Поменяйте `name` на любой тип из каталога выше (и при желании язык/сложность).

In [ ]:
name = "series_parallel_network"   # ← любой тип из ALL_TASK_GENERATORS

show_full(name, language="ru", difficulty=6)
show_full(name, language="en", difficulty=6)